# Train ANN posture model on Kaggle - posture_data_2fps.csv

Notebook nay train model ANN tu file `posture_data_2fps.csv` de so sanh voi lan train truoc.

Output se duoc luu vao `/kaggle/working`:

- `ann_best.keras`
- `scaler.pkl`
- `metrics.txt`
- `classification_report.txt`
- `confusion_matrix.csv`
- `training_curves.png`
- `confusion_matrix.png`
- `label_distribution.png`
- `normalized_confusion_matrix.png`
- `roc_curve.png`
- `precision_recall_curve.png`
- `threshold_analysis.png`
- `prediction_probability_histogram.png`

In [ ]:
import os
import io
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    auc,
    precision_recall_curve,
    average_precision_score,
)
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print('TensorFlow version:', tf.__version__)

## 1. Load dataset

Hay add dataset co file `posture_data_2fps.csv` vao Kaggle Notebook truoc khi chay.

In [ ]:
DATASET_FILENAME = 'posture_data_2fps.csv'
KAGGLE_INPUT_DIR = '/kaggle/input'
OUTPUT_DIR = '/kaggle/working'
os.makedirs(OUTPUT_DIR, exist_ok=True)

csv_path = None
for root, dirs, files in os.walk(KAGGLE_INPUT_DIR):
    if DATASET_FILENAME in files:
        csv_path = os.path.join(root, DATASET_FILENAME)
        break

if csv_path is None:
    raise FileNotFoundError(
        f'Khong tim thay {DATASET_FILENAME} trong {KAGGLE_INPUT_DIR}. '
        'Hay kiem tra phan Add Data cua Kaggle Notebook.'
    )

print('CSV path:', csv_path)
df = pd.read_csv(csv_path)

print('Dataset shape:', df.shape)
display(df.head())
print('\nDataframe info:')
df.info()
print('\nMissing values:', int(df.isnull().sum().sum()))
print('\nLabel distribution:')
print(df['label'].value_counts().sort_index())

## 2. Validate dataset

CSV phai gom 99 cot feature landmark va 1 cot `label`. Label:

- `0`: correct posture
- `1`: incorrect posture

In [ ]:
if 'label' not in df.columns:
    raise ValueError("Dataset phai co cot 'label'.")

num_features = df.drop(columns=['label']).shape[1]
if num_features != 99:
    raise ValueError(f'So feature khong dung. Ky vong 99, nhan duoc {num_features}.')

if df.isnull().sum().sum() > 0:
    rows_before = len(df)
    df = df.dropna().reset_index(drop=True)
    print(f'Da loai bo {rows_before - len(df)} dong co missing values.')

df['label'] = df['label'].astype(int)
unique_labels = sorted(df['label'].unique().tolist())
if set(unique_labels) != {0, 1}:
    raise ValueError(f'Label chi duoc gom 0 va 1. Hien co: {unique_labels}')

X = df.drop(columns=['label']).astype(np.float32)
y = df['label'].astype(int)

print('X shape:', X.shape)
print('y shape:', y.shape)
print('\nLabel percentage:')
print((y.value_counts(normalize=True).sort_index() * 100).round(2).astype(str) + '%')

## 3. Split train / validation / test

Ty le chia:

- Train: 70%
- Validation: 15%
- Test: 15%

In [ ]:
X_train_temp, X_test, y_train_temp, y_test = train_test_split(
    X,
    y,
    test_size=0.15,
    random_state=SEED,
    stratify=y,
)

validation_size = 0.15 / 0.85
X_train, X_val, y_train, y_val = train_test_split(
    X_train_temp,
    y_train_temp,
    test_size=validation_size,
    random_state=SEED,
    stratify=y_train_temp,
)

print('X_train:', X_train.shape)
print('X_val  :', X_val.shape)
print('X_test :', X_test.shape)

for name, labels in [('Train', y_train), ('Validation', y_val), ('Test', y_test)]:
    print(f'\n{name} label distribution:')
    print(labels.value_counts().sort_index())

## 4. Scale features

`StandardScaler` chi fit tren train set de tranh data leakage.

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print('Mean X_train_scaled:', np.mean(X_train_scaled).round(6))
print('Std  X_train_scaled:', np.std(X_train_scaled).round(6))

## 5. Build ANN model

In [ ]:
input_dim = X_train_scaled.shape[1]

model = Sequential([
    Input(shape=(input_dim,)),
    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.25),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid'),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy'],
)

model.summary()

## 6. Train model

In [ ]:
best_model_path = os.path.join(OUTPUT_DIR, 'ann_best.keras')

callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=15,
        restore_best_weights=True,
    ),
    ModelCheckpoint(
        filepath=best_model_path,
        monitor='val_loss',
        save_best_only=True,
        mode='min',
        verbose=1,
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1,
    ),
]

classes = np.array([0, 1])
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train,
)
class_weight_dict = {int(label): float(weight) for label, weight in zip(classes, class_weights)}
print('Class weight:', class_weight_dict)

EPOCHS = 100
BATCH_SIZE = 32

history = model.fit(
    X_train_scaled,
    y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    class_weight=class_weight_dict,
    verbose=1,
)

## 7. Save training curves

In [ ]:
curves_path = os.path.join(OUTPUT_DIR, 'training_curves.png')

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(history.history['loss'], label='Training Loss')
axes[0].plot(history.history['val_loss'], label='Validation Loss')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(history.history['accuracy'], label='Training Accuracy')
axes[1].plot(history.history['val_accuracy'], label='Validation Accuracy')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

fig.tight_layout()
fig.savefig(curves_path, dpi=150)
plt.show()

print('Saved training curves:', curves_path)

## 8. Evaluate on test set

In [ ]:
if os.path.exists(best_model_path):
    model = tf.keras.models.load_model(best_model_path)
    print('Loaded best model:', best_model_path)
else:
    model.save(best_model_path)
    print('Saved current model:', best_model_path)

THRESHOLD = 0.5
y_prob = model.predict(X_test_scaled, verbose=0)
y_pred = (y_prob >= THRESHOLD).astype(int).ravel()

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, pos_label=1, zero_division=0)
recall = recall_score(y_test, y_pred, pos_label=1, zero_division=0)
f1 = f1_score(y_test, y_pred, pos_label=1, zero_division=0)
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
y_score = y_prob.ravel()
fpr, tpr, roc_thresholds = roc_curve(y_test, y_score, pos_label=1)
roc_auc = auc(fpr, tpr)
pr_precision, pr_recall, pr_thresholds = precision_recall_curve(y_test, y_score, pos_label=1)
average_precision = average_precision_score(y_test, y_score)
report = classification_report(
    y_test,
    y_pred,
    labels=[0, 1],
    target_names=['0 - Correct', '1 - Incorrect'],
    zero_division=0,
)

print('Test Accuracy :', round(accuracy, 4))
print('Test Precision:', round(precision, 4))
print('Test Recall   :', round(recall, 4))
print('Test F1-score :', round(f1, 4))
print('ROC AUC       :', round(roc_auc, 4))
print('Avg Precision :', round(average_precision, 4))
print('\nConfusion matrix [[TN, FP], [FN, TP]]:')
print(cm)
print('\nClassification report:')
print(report)

## 9. Save confusion matrix image

In [ ]:
cm_image_path = os.path.join(OUTPUT_DIR, 'confusion_matrix.png')

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap='Blues')

class_names = ['0 - Correct', '1 - Incorrect']
ax.set_xticks(np.arange(len(class_names)))
ax.set_yticks(np.arange(len(class_names)))
ax.set_xticklabels(class_names)
ax.set_yticklabels(class_names)
ax.set_xlabel('Predicted label')
ax.set_ylabel('True label')
ax.set_title('Confusion Matrix')

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        text_color = 'white' if cm[i, j] > cm.max() / 2 else 'black'
        ax.text(j, i, cm[i, j], ha='center', va='center', color=text_color, fontsize=12)

fig.colorbar(im)
fig.tight_layout()
fig.savefig(cm_image_path, dpi=150)
plt.show()

print('Saved confusion matrix image:', cm_image_path)

## 10. Additional evaluation charts for report

In [ ]:
label_distribution_path = os.path.join(OUTPUT_DIR, 'label_distribution.png')
cm_normalized_image_path = os.path.join(OUTPUT_DIR, 'normalized_confusion_matrix.png')
roc_curve_path = os.path.join(OUTPUT_DIR, 'roc_curve.png')
precision_recall_curve_path = os.path.join(OUTPUT_DIR, 'precision_recall_curve.png')
threshold_analysis_path = os.path.join(OUTPUT_DIR, 'threshold_analysis.png')
probability_histogram_path = os.path.join(OUTPUT_DIR, 'prediction_probability_histogram.png')
threshold_metrics_csv_path = os.path.join(OUTPUT_DIR, 'threshold_metrics.csv')

# 1. Label distribution
label_counts = y.value_counts().sort_index()
fig, ax = plt.subplots(figsize=(6, 4.5))
bars = ax.bar(['0 - Correct', '1 - Incorrect'], label_counts.values, color=['#22c55e', '#ef4444'])
ax.set_title('Label Distribution')
ax.set_xlabel('Class')
ax.set_ylabel('Number of samples')
ax.grid(axis='y', alpha=0.25)
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, height, str(int(height)), ha='center', va='bottom')
fig.tight_layout()
fig.savefig(label_distribution_path, dpi=150)
plt.show()

# 2. Normalized confusion matrix
cm_normalized = np.divide(
    cm.astype(float),
    cm.sum(axis=1, keepdims=True),
    out=np.zeros_like(cm, dtype=float),
    where=cm.sum(axis=1, keepdims=True) != 0,
)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm_normalized, cmap='Blues', vmin=0, vmax=1)
class_names = ['0 - Correct', '1 - Incorrect']
ax.set_xticks(np.arange(len(class_names)))
ax.set_yticks(np.arange(len(class_names)))
ax.set_xticklabels(class_names)
ax.set_yticklabels(class_names)
ax.set_xlabel('Predicted label')
ax.set_ylabel('True label')
ax.set_title('Normalized Confusion Matrix')
for i in range(cm_normalized.shape[0]):
    for j in range(cm_normalized.shape[1]):
        value = cm_normalized[i, j]
        text_color = 'white' if value > 0.5 else 'black'
        ax.text(j, i, f'{value:.2%}', ha='center', va='center', color=text_color, fontsize=12)
fig.colorbar(im)
fig.tight_layout()
fig.savefig(cm_normalized_image_path, dpi=150)
plt.show()

# 3. ROC curve
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, label=f'ROC AUC = {roc_auc:.4f}', color='#2563eb', linewidth=2)
ax.plot([0, 1], [0, 1], linestyle='--', color='#6b7280', label='Random baseline')
ax.set_title('ROC Curve')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.grid(True, alpha=0.3)
ax.legend(loc='lower right')
fig.tight_layout()
fig.savefig(roc_curve_path, dpi=150)
plt.show()

# 4. Precision-Recall curve
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(pr_recall, pr_precision, label=f'AP = {average_precision:.4f}', color='#7c3aed', linewidth=2)
ax.set_title('Precision-Recall Curve')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.grid(True, alpha=0.3)
ax.legend(loc='lower left')
fig.tight_layout()
fig.savefig(precision_recall_curve_path, dpi=150)
plt.show()

# 5. Threshold analysis
threshold_values = np.arange(0.05, 0.96, 0.05)
threshold_rows = []
for threshold in threshold_values:
    threshold_pred = (y_score >= threshold).astype(int)
    threshold_rows.append({
        'threshold': threshold,
        'accuracy': accuracy_score(y_test, threshold_pred),
        'precision': precision_score(y_test, threshold_pred, pos_label=1, zero_division=0),
        'recall': recall_score(y_test, threshold_pred, pos_label=1, zero_division=0),
        'f1': f1_score(y_test, threshold_pred, pos_label=1, zero_division=0),
    })
threshold_df = pd.DataFrame(threshold_rows)
threshold_df.to_csv(threshold_metrics_csv_path, index=False)
best_f1_row = threshold_df.loc[threshold_df['f1'].idxmax()]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(threshold_df['threshold'], threshold_df['precision'], marker='o', label='Precision')
ax.plot(threshold_df['threshold'], threshold_df['recall'], marker='o', label='Recall')
ax.plot(threshold_df['threshold'], threshold_df['f1'], marker='o', label='F1-score')
ax.axvline(THRESHOLD, color='#111827', linestyle='--', linewidth=1.5, label=f'Current threshold = {THRESHOLD:.2f}')
ax.axvline(best_f1_row['threshold'], color='#f59e0b', linestyle=':', linewidth=2, label=f"Best F1 threshold = {best_f1_row['threshold']:.2f}")
ax.set_title('Precision, Recall and F1 by Threshold')
ax.set_xlabel('Decision threshold')
ax.set_ylabel('Score')
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)
ax.legend(loc='best')
fig.tight_layout()
fig.savefig(threshold_analysis_path, dpi=150)
plt.show()

# 6. Prediction probability histogram
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(y_score[y_test.to_numpy() == 0], bins=30, alpha=0.7, label='True 0 - Correct', color='#22c55e')
ax.hist(y_score[y_test.to_numpy() == 1], bins=30, alpha=0.7, label='True 1 - Incorrect', color='#ef4444')
ax.axvline(THRESHOLD, color='#111827', linestyle='--', linewidth=1.5, label=f'Threshold = {THRESHOLD:.2f}')
ax.set_title('Predicted Probability Distribution')
ax.set_xlabel('Predicted probability of incorrect posture')
ax.set_ylabel('Number of test samples')
ax.grid(axis='y', alpha=0.25)
ax.legend(loc='best')
fig.tight_layout()
fig.savefig(probability_histogram_path, dpi=150)
plt.show()

print('Saved additional charts:')
for path in [
    label_distribution_path,
    cm_normalized_image_path,
    roc_curve_path,
    precision_recall_curve_path,
    threshold_analysis_path,
    probability_histogram_path,
    threshold_metrics_csv_path,
]:
    print(f'- {path} | exists: {os.path.exists(path)}')

## 11. Save artifacts and reports

In [ ]:
scaler_path = os.path.join(OUTPUT_DIR, 'scaler.pkl')
metrics_path = os.path.join(OUTPUT_DIR, 'metrics.txt')
report_path = os.path.join(OUTPUT_DIR, 'classification_report.txt')
cm_csv_path = os.path.join(OUTPUT_DIR, 'confusion_matrix.csv')

joblib.dump(scaler, scaler_path)

cm_df = pd.DataFrame(
    cm,
    index=['true_0_correct', 'true_1_incorrect'],
    columns=['pred_0_correct', 'pred_1_incorrect'],
)
cm_df.to_csv(cm_csv_path)

with open(report_path, 'w', encoding='utf-8') as f:
    f.write(report)

summary_buffer = io.StringIO()
model.summary(print_fn=lambda line: summary_buffer.write(line + '\n'))
model_summary_text = summary_buffer.getvalue()

metrics_text = f'''Posture Detection ANN Metrics - posture_data_2fps
====================================================

Dataset path: {csv_path}
Dataset shape: {df.shape}

Train shape: {X_train.shape}
Validation shape: {X_val.shape}
Test shape: {X_test.shape}

Class distribution:
{y.value_counts().sort_index().to_string()}

Class weight:
{class_weight_dict}

Decision threshold: {THRESHOLD:.4f}

Test Accuracy: {accuracy:.6f}
Test Precision: {precision:.6f}
Test Recall: {recall:.6f}
Test F1-score: {f1:.6f}
ROC AUC: {roc_auc:.6f}
Average Precision: {average_precision:.6f}
Best F1 threshold from threshold analysis: {best_f1_row['threshold']:.4f}
Best F1 score from threshold analysis: {best_f1_row['f1']:.6f}

Confusion matrix [[TN, FP], [FN, TP]]:
{cm}

Classification report:
{report}

Model architecture summary:
{model_summary_text}
'''

with open(metrics_path, 'w', encoding='utf-8') as f:
    f.write(metrics_text)

output_files = [
    best_model_path,
    scaler_path,
    metrics_path,
    report_path,
    cm_csv_path,
    curves_path,
    cm_image_path,
    label_distribution_path,
    cm_normalized_image_path,
    roc_curve_path,
    precision_recall_curve_path,
    threshold_analysis_path,
    probability_histogram_path,
    threshold_metrics_csv_path,
]

print('Output files:')
for path in output_files:
    print(f'- {path} | exists: {os.path.exists(path)}')

## 12. Download note

Sau khi train xong tren Kaggle, vao phan Output cua notebook va tai ve cac file can so sanh:

- `metrics.txt`
- `training_curves.png`
- `confusion_matrix.png`
- `normalized_confusion_matrix.png`
- `label_distribution.png`
- `roc_curve.png`
- `precision_recall_curve.png`
- `threshold_analysis.png`
- `prediction_probability_histogram.png`
- `threshold_metrics.csv`
- `classification_report.txt`
- `ann_best.keras`
- `scaler.pkl`

Neu model nay tot hon, copy `ann_best.keras` va `scaler.pkl` ve project local vao thu muc `models/`.